# Databricks AI Functions for Document Processing

**Your Volume:** `/Volumes/infusion_center_demo/document_ai/raw_documents`

This notebook demonstrates 4 powerful AI functions that work together to process documents:

1. **`ai_parse_document()`** - Extract structured content from PDFs/images
2. **`ai_classify()`** - Classify documents into categories
3. **`ai_extract()`** - Extract specific fields with typed schemas
4. **`ai_prep_search()`** - Prepare documents for semantic search

**Key Insight:** These functions are composable - the output of one feeds into the next!

## 1. `ai_parse_document()` - Read and Extract Text from PDFs

**What it does:** Converts unstructured documents (PDFs, images, Office files) into structured data that AI can understand.

**Key Points:**
* Takes BINARY content from files (not file paths)
* Returns a VARIANT with structured elements (text, tables, figures)
* Always use `MAP('version', '2.0')` for best results
* Preserves document structure (titles, sections, tables)

In [0]:
%sql
-- Parse all PDFs from your volume
-- READ_FILES with format => 'binaryFile' loads the raw bytes

SELECT
  path,
  ai_parse_document(content, MAP('version', '2.0')) AS parsed_content
FROM READ_FILES(
  '/Volumes/infusion_center_demo/document_ai/raw_documents/',
  format => 'binaryFile'
)
--LIMIT 3

path parsed_content dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC6_LabResult_CBC_Critical_ANC.pdf {"document":{"elements":[{"bbox":[{"coord":[163,160,533,204],"page_id":0}],"confidence":0.9996,"content":"Quest Diagnostics","description":null,"id":0,"type":"text"},{"bbox":[{"coord":[163,201,417,228],"page_id":0}],"confidence":0.9983,"content":"Clinical Laboratory Results","description":null,"id":1,"type":"text"},{"bbox":[{"coord":[1099,155,1535,182],"page_id":0}],"confidence":1,"content":"Phone: 1-800-222-0446 | Fax: 1-800-422-1285","description":null,"id":2,"type":"text"},{"bbox":[{"coord":[163,250,526,275],"page_id":0}],"confidence":0.9999,"content":"1001 Main Street, Teterboro NJ 07608","description":null,"id":3,"type":"text"},{"bbox":[{"coord":[387,334,1313,371],"page_id":0}],"confidence":0.9965,"content":"LABORATORY REPORT — CBC WITH DIFFERENTIAL","description":null,"id":4,"type":"title"},{"bbox":[{"coord":[163,400,1057,778],"page_id":0}],"confidence":0.9955,"content":" Patient Name: Alice M. Nguyen Date of Birth: September 5, 1962 MRN: QDX-112233 Ordering Provider: Dr. Rachel Kim, MD — Hematology/Oncology Specimen Collected: January 19, 2025 — 07:30 AM Report Date: January 19, 2025 — 10:15 AM Accession #: 2025-01-19-QD-884421 ","description":null,"id":5,"type":"table"},{"bbox":[{"coord":[199,823,1501,1375],"page_id":0}],"confidence":0.9925,"content":" TEST RESULT FLAG REFERENCE RANGE UNITS WBC 2.8 LOW ↓ 4.5 – 11.0 K/uL RBC 3.62 LOW ↓ 4.20 – 5.40 M/uL Hemoglobin (HGB) 9.4 LOW ↓ 12.0 – 16.0 g/dL Hematocrit (HCT) 28.1 LOW ↓ 36.0 – 46.0 % Platelets (PLT) 142 150 – 400 K/uL ANC (Neutrophils) 1.1 CRITICAL ↓↓ 1.8 – 7.7 K/uL Lymphocytes 1.3 LOW ↓ 1.0 – 4.8 K/uL Monocytes 0.3 0.2 – 0.9 K/uL ","description":null,"id":6,"type":"table"},{"bbox":[{"coord":[0,1424,376,1456],"page_id":0}],"confidence":0.9538,"content":"emotherapy threshold ≥ 1.5 K/uL)","description":null,"id":7,"type":"page_footer"},{"bbox":[{"coord":[875,1427,1700,1458],"page_id":0}],"confidence":0.9998,"content":"Ordering provider Dr. Kim notified at 10:22 AM. RECOMMENDATION: Hold ch","description":null,"id":8,"type":"page_footer"}],"pages":[{"id":0,"image_uri":null}]},"error_status":null,"metadata":{"file_metadata":null,"id":"b3265db3-0bf3-4b30-a9e2-a6b48ec923ae","version":"2.0"}} dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC5_AdverseEvent_Grade3_Pembrolizumab.pdf {"document":{"elements":[{"bbox":[{"coord":[1135,155,1535,184],"page_id":0}],"confidence":1,"content":"Phone: 214-820-3300 | Fax: 214-820-3391","description":null,"id":0,"type":"page_header"},{"bbox":[{"coord":[165,160,778,201],"page_id":0}],"confidence":1,"content":"Infusion Center of North Texas","description":null,"id":1,"type":"title"},{"bbox":[{"coord":[163,201,467,231],"page_id":0}],"confidence":0.9999,"content":"Adverse Event / Incident Report","description":null,"id":2,"type":"text"},{"bbox":[{"coord":[163,248,632,277],"page_id":0}],"confidence":0.9998,"content":"3500 Gaston Avenue, Suite 200, Dallas TX 75246","description":null,"id":3,"type":"text"},{"bbox":[{"coord":[338,339,1364,378],"page_id":0}],"confidence":0.9808,"content":"ADVERSE EVENT REPORT — NURSING DOCUMENTATION","description":null,"id":4,"type":"section_header"},{"bbox":[{"coord":[0,415,298,449],"page_id":0}],"confidence":0.9371,"content":"harmacist review required","description":null,"id":5,"type":"text"},{"bbox":[{"coord":[875,417,1588,452],"page_id":0}],"confidence":0.9991,"content":"Physician notified. Mandatory pharmacovigilance reporting initiated.","description":null,"id":6,"type":"text"},{"bbox":[{"coord":[161,506,1260,953],"page_id":0}],"confidence":0.9956,"content":" Report Date/Time: January 18, 2025 — 10:42 AM Patient Name: Elena V. Kowalski Date of Birth: February 14, 1979 MRN: ICNT-334821 Attending Nurse: Jennifer Huang, RN, OCN Drug: Pembrolizumab (Keytruda) 200 mg IV — Cycle 4, Day 1 Diagnosis: C34.12 — Malignant neoplasm of upper lobe, left bronchus/lung Reaction Onset: 10:38 AM — 

In [0]:
# PySpark equivalent - same logic, different syntax

from pyspark.sql.functions import col

# Read binary files from the volume
df = spark.read.format("binaryFile") \
    .load("/Volumes/infusion_center_demo/document_ai/raw_documents/")

# Parse documents using ai_parse_document
parsed_df = df.selectExpr(
    "path",
    "ai_parse_document(content, map('version', '2.0')) AS parsed_content"
)

# Show first 3 results
display(parsed_df.limit(3))

path parsed_content dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC6_LabResult_CBC_Critical_ANC.pdf {"document":{"elements":[{"bbox":[{"coord":[163,160,533,204],"page_id":0}],"confidence":0.9996,"content":"Quest Diagnostics","description":null,"id":0,"type":"text"},{"bbox":[{"coord":[163,201,417,228],"page_id":0}],"confidence":0.9983,"content":"Clinical Laboratory Results","description":null,"id":1,"type":"text"},{"bbox":[{"coord":[1099,155,1535,182],"page_id":0}],"confidence":1,"content":"Phone: 1-800-222-0446 | Fax: 1-800-422-1285","description":null,"id":2,"type":"text"},{"bbox":[{"coord":[163,250,526,275],"page_id":0}],"confidence":0.9999,"content":"1001 Main Street, Teterboro NJ 07608","description":null,"id":3,"type":"text"},{"bbox":[{"coord":[387,334,1313,371],"page_id":0}],"confidence":0.9962,"content":"LABORATORY REPORT — CBC WITH DIFFERENTIAL","description":null,"id":4,"type":"title"},{"bbox":[{"coord":[163,400,1057,778],"page_id":0}],"confidence":0.9955,"content":" Patient Name: Alice M. Nguyen Date of Birth: September 5, 1962 MRN: QDX-112233 Ordering Provider: Dr. Rachel Kim, MD — Hematology/Oncology Specimen Collected: January 19, 2025 — 07:30 AM Report Date: January 19, 2025 — 10:15 AM Accession #: 2025-01-19-QD-884421 ","description":null,"id":5,"type":"table"},{"bbox":[{"coord":[199,823,1501,1375],"page_id":0}],"confidence":0.9926,"content":" TEST RESULT FLAG REFERENCE RANGE UNITS WBC 2.8 LOW ↓ 4.5 – 11.0 K/uL RBC 3.62 LOW ↓ 4.20 – 5.40 M/uL Hemoglobin (HGB) 9.4 LOW ↓ 12.0 – 16.0 g/dL Hematocrit (HCT) 28.1 LOW ↓ 36.0 – 46.0 % Platelets (PLT) 142 150 – 400 K/uL ANC (Neutrophils) 1.1 CRITICAL ↓↓ 1.8 – 7.7 K/uL Lymphocytes 1.3 LOW ↓ 1.0 – 4.8 K/uL Monocytes 0.3 0.2 – 0.9 K/uL ","description":null,"id":6,"type":"table"},{"bbox":[{"coord":[0,1424,376,1456],"page_id":0}],"confidence":0.9536,"content":"emotherapy threshold ≥ 1.5 K/uL)","description":null,"id":7,"type":"page_footer"},{"bbox":[{"coord":[875,1427,1700,1458],"page_id":0}],"confidence":0.9998,"content":"Ordering provider Dr. Kim notified at 10:22 AM. RECOMMENDATION: Hold ch","description":null,"id":8,"type":"page_footer"}],"pages":[{"id":0,"image_uri":null}]},"error_status":null,"metadata":{"file_metadata":null,"id":"22379946-df29-46b7-92db-a0a984c522ed","version":"2.0"}} dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC5_AdverseEvent_Grade3_Pembrolizumab.pdf {"document":{"elements":[{"bbox":[{"coord":[1135,155,1533,182],"page_id":0}],"confidence":1,"content":"Phone: 214-820-3300 | Fax: 214-820-3391","description":null,"id":0,"type":"page_header"},{"bbox":[{"coord":[165,162,776,199],"page_id":0}],"confidence":1,"content":"Infusion Center of North Texas","description":null,"id":1,"type":"title"},{"bbox":[{"coord":[165,204,463,228],"page_id":0}],"confidence":0.9999,"content":"Adverse Event / Incident Report","description":null,"id":2,"type":"text"},{"bbox":[{"coord":[165,250,630,275],"page_id":0}],"confidence":0.9997,"content":"3500 Gaston Avenue, Suite 200, Dallas TX 75246","description":null,"id":3,"type":"text"},{"bbox":[{"coord":[340,341,1360,376],"page_id":0}],"confidence":0.9514,"content":"ADVERSE EVENT REPORT — NURSING DOCUMENTATION","description":null,"id":4,"type":"section_header"},{"bbox":[{"coord":[0,417,294,447],"page_id":0}],"confidence":0.9031,"content":"harmacist review required","description":null,"id":5,"type":"text"},{"bbox":[{"coord":[877,420,1584,449],"page_id":0}],"confidence":0.9996,"content":"Physician notified. Mandatory pharmacovigilance reporting initiated.","description":null,"id":6,"type":"text"},{"bbox":[{"coord":[163,508,1256,950],"page_id":0}],"confidence":0.995,"content":" Report Date/Time: January 18, 2025 — 10:42 AM Patient Name: Elena V. Kowalski Date of Birth: February 14, 1979 MRN: ICNT-334821 Attending Nurse: Jennifer Huang, RN, OCN Drug: Pembrolizumab (Keytruda) 200 mg IV — Cycle 4, Day 1 Diagnosis: C34.12 — Malignant neoplasm of upper lobe, left bronchus/lung Reaction Onset: 10:38 AM — a

## 2. `ai_classify()` - Identify Document Types

**What it does:** Automatically categorizes documents into labels you define.

**Key Points:**
* Pass the VARIANT from `ai_parse_document()` directly (don't flatten to text!)
* Use JSON object format for labels with descriptions (improves accuracy)
* Always use `MAP('version', '2.0')` for VARIANT support
* Returns `{"response": ["label"], "error_message": null}`

In [0]:
%sql
-- Chain ai_parse_document → ai_classify
-- Classify medical/healthcare documents into categories

WITH parsed_docs AS (
  SELECT
    path,
    ai_parse_document(content, MAP('version', '2.0')) AS parsed_content
  FROM READ_FILES(
    '/Volumes/infusion_center_demo/document_ai/raw_documents/',
    format => 'binaryFile'
  )
)
SELECT
  path,
  ai_classify(
    parsed_content,
    '{
      "patient_record": "Medical records, patient history, treatment notes",
      "prescription": "Medication prescriptions and pharmacy orders",
      "lab_report": "Lab results, test reports, diagnostic findings",
      "insurance_form": "Insurance claims, authorization forms, billing",
      "consent_form": "Informed consent, HIPAA forms, patient agreements"
    }',
    MAP(
      'version', '2.0',
      'instructions', 'Classify healthcare documents by their primary purpose.'
    )
  ) AS document_type
FROM parsed_docs
LIMIT 5

path,document_type
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC6_LabResult_CBC_Critical_ANC.pdf,"{""error_message"":null,""response"":[""lab_report""]}"
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC5_AdverseEvent_Grade3_Pembrolizumab.pdf,"{""error_message"":null,""response"":[""patient_record""]}"
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC3_Referral_RCHOP_Oncology_Routine.pdf,"{""error_message"":null,""response"":[""patient_record""]}"
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC2_PriorAuth_DENIAL_Rituximab_AETNA.pdf,"{""error_message"":null,""response"":[""insurance_form""]}"
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC1_PriorAuth_APPROVAL_Herceptin_BCBS.pdf,"{""error_message"":null,""response"":[""insurance_form""]}"


In [0]:
# PySpark chaining: parse → classify

from pyspark.sql.functions import expr

# Read and parse
df = spark.read.format("binaryFile") \
    .load("/Volumes/infusion_center_demo/document_ai/raw_documents/")

parsed_df = df.selectExpr(
    "path",
    "ai_parse_document(content, map('version', '2.0')) AS parsed_content"
)

# Classify
classified_df = parsed_df.selectExpr(
    "path",
    """ai_classify(
        parsed_content,
        '{
          "patient_record": "Medical records, patient history, treatment notes",
          "prescription": "Medication prescriptions and pharmacy orders",
          "lab_report": "Lab results, test reports, diagnostic findings",
          "insurance_form": "Insurance claims, authorization forms, billing",
          "consent_form": "Informed consent, HIPAA forms, patient agreements"
        }',
        map(
          'version', '2.0',
          'instructions', 'Classify healthcare documents by their primary purpose.'
        )
      ) AS document_type"""
)

display(classified_df.limit(5))

path,document_type
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC6_LabResult_CBC_Critical_ANC.pdf,"{""error_message"":null,""response"":[""lab_report""]}"
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC5_AdverseEvent_Grade3_Pembrolizumab.pdf,"{""error_message"":null,""response"":[""patient_record""]}"
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC3_Referral_RCHOP_Oncology_Routine.pdf,"{""error_message"":null,""response"":[""patient_record""]}"
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC2_PriorAuth_DENIAL_Rituximab_AETNA.pdf,"{""error_message"":null,""response"":[""insurance_form""]}"
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC1_PriorAuth_APPROVAL_Herceptin_BCBS.pdf,"{""error_message"":null,""response"":[""insurance_form""]}"


## 3. `ai_extract()` - Extract Specific Fields

**What it does:** Pulls out specific information from documents and structures it with data types.

**Key Points:**
* Pass the VARIANT from `ai_parse_document()` directly
* Define a JSON schema with field names, types, and descriptions
* Supports nested objects, arrays, and type validation
* Types: `string`, `integer`, `number`, `boolean`, `enum`, `array`, `object`
* Returns `{"response": {...}, "error_message": null}`

In [0]:
%sql
-- Chain ai_parse_document → ai_extract
-- Extract specific fields from patient records

WITH parsed_docs AS (
  SELECT
    path,
    ai_parse_document(content, MAP('version', '2.0')) AS parsed_content
  FROM READ_FILES(
    '/Volumes/infusion_center_demo/document_ai/raw_documents/',
    format => 'binaryFile'
  )
)
SELECT
  path,
  ai_extract(
    parsed_content,
    '{
      "patient_name": {
        "type": "string",
        "description": "Full name of the patient"
      },
      "patient_id": {
        "type": "string",
        "description": "Medical record number or patient ID"
      },
      "date_of_service": {
        "type": "string",
        "description": "Date in YYYY-MM-DD format"
      },
      "diagnosis": {
        "type": "string",
        "description": "Primary diagnosis or reason for visit"
      },
      "medications": {
        "type": "array",
        "description": "List of prescribed medications",
        "items": {
          "type": "object",
          "properties": {
            "name": {"type": "string"},
            "dosage": {"type": "string"},
            "frequency": {"type": "string"}
          }
        }
      }
    }',
    MAP(
      'version', '2.0',
      'instructions', 'Extract key patient information from medical documents.'
    )
  ) AS extracted_data
FROM parsed_docs


path,extracted_data
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC6_LabResult_CBC_Critical_ANC.pdf,"{""error_message"":null,""response"":{""date_of_service"":""2025-01-19"",""diagnosis"":null,""medications"":[],""patient_id"":""QDX-112233"",""patient_name"":""Alice M. Nguyen""}}"
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC5_AdverseEvent_Grade3_Pembrolizumab.pdf,"{""error_message"":null,""response"":{""date_of_service"":""2025-01-18"",""diagnosis"":""C34.12 — Malignant neoplasm of upper lobe, left bronchus/lung"",""medications"":[{""dosage"":""200 mg IV"",""frequency"":null,""name"":""Pembrolizumab""},{""dosage"":""0.3 mg IM"",""frequency"":null,""name"":""Epinephrine""},{""dosage"":""50 mg IV"",""frequency"":null,""name"":""Diphenhydramine""},{""dosage"":""125 mg IV"",""frequency"":null,""name"":""Methylprednisolone""}],""patient_id"":""ICNT-334821"",""patient_name"":""Elena V. Kowalski""}}"
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC3_Referral_RCHOP_Oncology_Routine.pdf,"{""error_message"":null,""response"":{""date_of_service"":""2025-01-17"",""diagnosis"":""C83.39 — Diffuse large B-cell lymphoma, multiple sites"",""medications"":[{""dosage"":""50mg IV"",""frequency"":null,""name"":""Diphenhydramine""},{""dosage"":""650mg PO"",""frequency"":null,""name"":""Acetaminophen""},{""dosage"":""8mg IV"",""frequency"":null,""name"":""Ondansetron""}],""patient_id"":""TXO-789456"",""patient_name"":""Robert A. Martinez""}}"
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC2_PriorAuth_DENIAL_Rituximab_AETNA.pdf,"{""error_message"":null,""response"":{""date_of_service"":""2025-01-16"",""diagnosis"":""M05.79 — Rheumatoid arthritis with rheumatoid factor, multiple sites"",""medications"":[{""dosage"":""1000 mg"",""frequency"":null,""name"":""Rituximab (Rituxan)""}],""patient_id"":""W8834521109"",""patient_name"":""James R. Holloway""}}"
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC1_PriorAuth_APPROVAL_Herceptin_BCBS.pdf,"{""error_message"":null,""response"":{""date_of_service"":""2025-01-14"",""diagnosis"":""C50.011 — Malignant neoplasm of central portion, right female breast"",""medications"":[{""dosage"":""440 mg"",""frequency"":""12 infusion cycles"",""name"":""Trastuzumab (Herceptin)""},{""dosage"":null,""frequency"":null,""name"":""diphenhydramine""},{""dosage"":null,""frequency"":null,""name"":""acetaminophen""}],""patient_id"":""XYY234567890"",""patient_name"":""Sarah J. Worthington""}}"
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC4_Referral_Tocilizumab_STAT_Rheumatology.pdf,"{""error_message"":null,""response"":{""date_of_service"":""2025-01-21"",""diagnosis"":""M06.09 — Rheumatoid arthritis without rheumatoid factor, multiple sites"",""medications"":[{""dosage"":""8 mg/kg IV"",""frequency"":null,""name"":""Tocilizumab (Actemra)""}],""patient_id"":""DRA-556677"",""patient_name"":""Marcus T. Williams""}}"
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC8_AdverseEvent_Grade2_Nausea_Oxaliplatin.pdf,"{""error_message"":null,""response"":{""date_of_service"":""2025-01-22"",""diagnosis"":""C18.7 — Malignant neoplasm of sigmoid colon"",""medications"":[{""dosage"":""8 mg IV"",""frequency"":null,""name"":""Ondansetron""},{""dosage"":""10 mg oral"",""frequency"":null,""name"":""Prochlorperazine""}],""patient_id"":""ICNT-228844"",""patient_name"":""Patricia M. O'Brien""}}"
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC7_InsuranceCard_UHC_ChoicePlus.pdf,"{""error_message"":null,""response"":{""date_of_service"":null,""diagnosis"":null,""medications"":[],""patient_id"":""W1234567890"",""patient_name"":""Thomas B. Reynolds""}}"


In [0]:
# PySpark chaining: parse → extract

# Read and parse
df = spark.read.format("binaryFile") \
    .load("/Volumes/infusion_center_demo/document_ai/raw_documents/")

parsed_df = df.selectExpr(
    "path",
    "ai_parse_document(content, map('version', '2.0')) AS parsed_content"
)

# Extract structured fields
extracted_df = parsed_df.selectExpr(
    "path",
    """ai_extract(
        parsed_content,
        '{
          "patient_name": {
            "type": "string",
            "description": "Full name of the patient"
          },
          "patient_id": {
            "type": "string",
            "description": "Medical record number or patient ID"
          },
          "date_of_service": {
            "type": "string",
            "description": "Date in YYYY-MM-DD format"
          },
          "diagnosis": {
            "type": "string",
            "description": "Primary diagnosis or reason for visit"
          },
          "medications": {
            "type": "array",
            "description": "List of prescribed medications",
            "items": {
              "type": "object",
              "properties": {
                "name": {"type": "string"},
                "dosage": {"type": "string"},
                "frequency": {"type": "string"}
              }
            }
          }
        }',
        map(
          'version', '2.0',
          'instructions', 'Extract key patient information from medical documents.'
        )
      ) AS extracted_data"""
)

display(extracted_df.limit(3))

path,extracted_data
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC6_LabResult_CBC_Critical_ANC.pdf,"{""error_message"":null,""response"":{""date_of_service"":""2025-01-19"",""diagnosis"":null,""medications"":[],""patient_id"":""QDX-112233"",""patient_name"":""Alice M. Nguyen""}}"
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC5_AdverseEvent_Grade3_Pembrolizumab.pdf,"{""error_message"":null,""response"":{""date_of_service"":""2025-01-18"",""diagnosis"":""C34.12 — Malignant neoplasm of upper lobe, left bronchus/lung"",""medications"":[{""dosage"":""0.3 mg"",""frequency"":""IM"",""name"":""Epinephrine""},{""dosage"":""50 mg"",""frequency"":""IV push"",""name"":""Diphenhydramine""},{""dosage"":""125 mg"",""frequency"":""IV"",""name"":""Methylprednisolone""},{""dosage"":""200 mg"",""frequency"":""IV"",""name"":""Pembrolizumab""}],""patient_id"":""ICNT-334821"",""patient_name"":""Elena V. Kowalski""}}"
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC3_Referral_RCHOP_Oncology_Routine.pdf,"{""error_message"":null,""response"":{""date_of_service"":""2025-01-17"",""diagnosis"":""C83.39 — Diffuse large B-cell lymphoma, multiple sites"",""medications"":[{""dosage"":""701 mg"",""frequency"":null,""name"":""Rituximab""},{""dosage"":""50 mg"",""frequency"":""IV"",""name"":""Diphenhydramine""},{""dosage"":""650 mg"",""frequency"":""PO"",""name"":""Acetaminophen""},{""dosage"":""8 mg"",""frequency"":""IV"",""name"":""Ondansetron""}],""patient_id"":""TXO-789456"",""patient_name"":""Robert A. Martinez""}}"


In [0]:
%sql
-- Extract client/provider name and phone number from documents
-- This is useful for extracting insurance provider or organization contact info

WITH parsed_docs AS (
  SELECT
    path,
    ai_parse_document(content, MAP('version', '2.0')) AS parsed_content
  FROM READ_FILES(
    '/Volumes/infusion_center_demo/document_ai/raw_documents/',
    format => 'binaryFile'
  )
)
SELECT
  path,
  ai_extract(
    parsed_content,
    '{
      "client_name": {
        "type": "string",
        "description": "Name of the organization, company, insurance provider, or business entity"
      },
      "phone_number": {
        "type": "string",
        "description": "Primary phone number in format like 1-800-XXX-XXXX or (XXX) XXX-XXXX"
      },
      "fax_number": {
        "type": "string",
        "description": "Fax number if available"
      },
      "address": {
        "type": "string",
        "description": "Mailing address including PO Box, street, city, state, and ZIP code"
      }
    }',
    MAP(
      'version', '2.0',
      'instructions', 'Extract organization/provider contact information from the document header or contact section.'
    )
  ) AS contact_info
FROM parsed_docs
LIMIT 5

path,contact_info
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC6_LabResult_CBC_Critical_ANC.pdf,"{""error_message"":null,""response"":{""address"":""1001 Main Street, Teterboro NJ 07608"",""client_name"":""Quest Diagnostics"",""fax_number"":""1-800-422-1285"",""phone_number"":""1-800-222-0446""}}"
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC5_AdverseEvent_Grade3_Pembrolizumab.pdf,"{""error_message"":null,""response"":{""address"":""3500 Gaston Avenue, Suite 200, Dallas TX 75246"",""client_name"":""Infusion Center of North Texas"",""fax_number"":""214-820-3391"",""phone_number"":""214-820-3300""}}"
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC3_Referral_RCHOP_Oncology_Routine.pdf,"{""error_message"":null,""response"":{""address"":""7777 Forest Lane, Suite C-700, Dallas TX 75230"",""client_name"":""Texas Oncology — Medical City Dallas"",""fax_number"":""214-370-1099"",""phone_number"":""214-370-1000""}}"
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC2_PriorAuth_DENIAL_Rituximab_AETNA.pdf,"{""error_message"":null,""response"":{""address"":""151 Farmington Avenue, Hartford CT 06156"",""client_name"":""Aetna Health Plans"",""fax_number"":""1-860-273-1822"",""phone_number"":""1-800-223-9870""}}"
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC1_PriorAuth_APPROVAL_Herceptin_BCBS.pdf,"{""error_message"":null,""response"":{""address"":""P.O. Box 660044, Dallas TX 75266"",""client_name"":""Blue Cross Blue Shield of Texas"",""fax_number"":""1-800-447-7828"",""phone_number"":""1-800-521-2227""}}"


In [0]:
%sql
-- Flatten the extracted contact information for easier querying
-- This shows how to access nested fields from the VARIANT response

WITH parsed_docs AS (
  SELECT
    path,
    ai_parse_document(content, MAP('version', '2.0')) AS parsed_content
  FROM READ_FILES(
    '/Volumes/infusion_center_demo/document_ai/raw_documents/',
    format => 'binaryFile'
  )
),
extracted_contacts AS (
  SELECT
    path,
    ai_extract(
      parsed_content,
      '{
        "client_name": {
          "type": "string",
          "description": "Name of the organization, company, insurance provider, or business entity"
        },
        "phone_number": {
          "type": "string",
          "description": "Primary phone number in format like 1-800-XXX-XXXX or (XXX) XXX-XXXX"
        },
        "fax_number": {
          "type": "string",
          "description": "Fax number if available"
        },
        "address": {
          "type": "string",
          "description": "Mailing address including PO Box, street, city, state, and ZIP code"
        }
      }',
      MAP(
        'version', '2.0',
        'instructions', 'Extract organization/provider contact information from the document header or contact section.'
      )
    ) AS contact_info
  FROM parsed_docs
)
SELECT
  REGEXP_EXTRACT(path, '([^/]+)\.pdf$', 1) AS document_name,
  contact_info:response:client_name::STRING AS client_name,
  contact_info:response:phone_number::STRING AS phone_number,
  contact_info:response:fax_number::STRING AS fax_number,
  contact_info:response:address::STRING AS address
FROM extracted_contacts
ORDER BY document_name

document_name,client_name,phone_number,fax_number,address
DOC1_PriorAuth_APPROVAL_Herceptin_BCBS,Blue Cross Blue Shield of Texas,1-800-521-2227,1-800-447-7828,"P.O. Box 660044, Dallas TX 75266"
DOC2_PriorAuth_DENIAL_Rituximab_AETNA,Aetna Health Plans,1-800-223-9870,1-860-273-1822,"151 Farmington Avenue, Hartford CT 06156"
DOC3_Referral_RCHOP_Oncology_Routine,Texas Oncology — Medical City Dallas,214-370-1000,214-370-1099,"7777 Forest Lane, Suite C-700, Dallas TX 75230"
DOC4_Referral_Tocilizumab_STAT_Rheumatology,Dallas Rheumatology Associates,214-553-0300,214-553-0399,"8144 Walnut Hill Lane, Suite 1200, Dallas TX 75231"
DOC5_AdverseEvent_Grade3_Pembrolizumab,Infusion Center of North Texas,214-820-3300,214-820-3391,"3500 Gaston Avenue, Suite 200, Dallas TX 75246"
DOC6_LabResult_CBC_Critical_ANC,Quest Diagnostics,1-800-222-0446,1-800-422-1285,"1001 Main Street, Teterboro NJ 07608"
DOC7_InsuranceCard_UHC_ChoicePlus,UnitedHealthcare,1-877-842-3210,null,"P.O. Box 30555, Salt Lake City UT 84130"
DOC8_AdverseEvent_Grade2_Nausea_Oxaliplatin,Infusion Center of North Texas,214-820-3300,214-820-3391,"3500 Gaston Avenue, Suite 200, Dallas TX 75246"


## 4. `ai_prep_search()` - Prepare Documents for Semantic Search

**What it does:** Breaks parsed documents into semantic chunks optimized for vector search and RAG (Retrieval Augmented Generation).

**Key Points:**
* Takes the VARIANT from `ai_parse_document()`
* Intelligently chunks documents by semantic meaning
* Each chunk can be embedded and stored in a vector database
* Essential for building search and Q&A systems over documents

In [0]:
%sql
-- Chain ai_parse_document → ai_prep_search
-- Create searchable chunks from documents

WITH parsed_docs AS (
  SELECT
    path,
    ai_parse_document(content, MAP('version', '2.0')) AS parsed_content
  FROM READ_FILES(
    '/Volumes/infusion_center_demo/document_ai/raw_documents/',
    format => 'binaryFile'
  )
)
SELECT
  path,
  ai_prep_search(parsed_content) AS search_chunks
FROM parsed_docs
LIMIT 3

path search_chunks dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC6_LabResult_CBC_Critical_ANC.pdf {"document":{"contents":[{"chunk_id":"030da2c89ceb4bd6a38b722d09a736e5_0","chunk_position":0,"chunk_to_embed":"The following passage represents a chunk of content from a document.\n - 'Content' contains raw document text\n - All other fields describe document context and hierchical information\n - For visual elements like images/charts, a summary is generated as part of 'Content'\n\n Document Title: LABORATORY REPORT — CBC WITH DIFFERENTIAL\n Page Header: \n Page Footer: Ordering provider Dr. Kim notified at 10:22 AM. RECOMMENDATION: Hold ch\n Section Header: \n Caption: \n Footnote: \n Page Number: \n\n Content:\n \nQuest Diagnostics\n\nClinical Laboratory Results\n\nPhone: 1-800-222-0446 | Fax: 1-800-422-1285\n\n1001 Main Street, Teterboro NJ 07608\n\n Patient Name: Alice M. Nguyen Date of Birth: September 5, 1962 MRN: QDX-112233 Ordering Provider: Dr. Rachel Kim, MD — Hematology/Oncology Specimen Collected: January 19, 2025 — 07:30 AM Report Date: January 19, 2025 — 10:15 AM Accession #: 2025-01-19-QD-884421 \n\n TEST RESULT FLAG REFERENCE RANGE UNITS WBC 2.8 LOW ↓ 4.5 – 11.0 K/uL RBC 3.62 LOW ↓ 4.20 – 5.40 M/uL Hemoglobin (HGB) 9.4 LOW ↓ 12.0 – 16.0 g/dL Hematocrit (HCT) 28.1 LOW ↓ 36.0 – 46.0 % Platelets (PLT) 142 150 – 400 K/uL ANC (Neutrophils) 1.1 CRITICAL ↓↓ 1.8 – 7.7 K/uL Lymphocytes 1.3 LOW ↓ 1.0 – 4.8 K/uL Monocytes 0.3 0.2 – 0.9 K/uL ","chunk_to_retrieve":"\nQuest Diagnostics\n\nClinical Laboratory Results\n\nPhone: 1-800-222-0446 | Fax: 1-800-422-1285\n\n1001 Main Street, Teterboro NJ 07608\n\n Patient Name: Alice M. Nguyen Date of Birth: September 5, 1962 MRN: QDX-112233 Ordering Provider: Dr. Rachel Kim, MD — Hematology/Oncology Specimen Collected: January 19, 2025 — 07:30 AM Report Date: January 19, 2025 — 10:15 AM Accession #: 2025-01-19-QD-884421 \n\n TEST RESULT FLAG REFERENCE RANGE UNITS WBC 2.8 LOW ↓ 4.5 – 11.0 K/uL RBC 3.62 LOW ↓ 4.20 – 5.40 M/uL Hemoglobin (HGB) 9.4 LOW ↓ 12.0 – 16.0 g/dL Hematocrit (HCT) 28.1 LOW ↓ 36.0 – 46.0 % Platelets (PLT) 142 150 – 400 K/uL ANC (Neutrophils) 1.1 CRITICAL ↓↓ 1.8 – 7.7 K/uL Lymphocytes 1.3 LOW ↓ 1.0 – 4.8 K/uL Monocytes 0.3 0.2 – 0.9 K/uL \n","pages":[{"image_uri":"","page_id":0}]}],"pages":[{"id":0,"image_uri":null}]},"error_status":null} dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC5_AdverseEvent_Grade3_Pembrolizumab.pdf {"document":{"contents":[{"chunk_id":"835e588c785349ab9ba16a44d71c5bc1_0","chunk_position":0,"chunk_to_embed":"The following passage represents a chunk of content from a document.\n - 'Content' contains raw document text\n - All other fields describe document context and hierchical information\n - For visual elements like images/charts, a summary is generated as part of 'Content'\n\n Document Title: Infusion Center of North Texas\n Page Header: Phone: 214-820-3300 | Fax: 214-820-3391\n Page Footer: \n Section Header: OUTCOME\n Caption: \n Footnote: \n Page Number: \n\n Content:\n \nAdverse Event / Incident Report\n\n3500 Gaston Avenue, Suite 200, Dallas TX 75246\n\nharmacist review required\n\nPhysician notified. Mandatory pharmacovigilance reporting initiated.\n\n Report Date/Time: January 18, 2025 — 10:42 AM Patient Name: Elena V. Kowalski Date of Birth: February 14, 1979 MRN: ICNT-334821 Attending Nurse: Jennifer Huang, RN, OCN Drug: Pembrolizumab (Keytruda) 200 mg IV — Cycle 4, Day 1 Diagnosis: C34.12 — Malignant neoplasm of upper lobe, left bronchus/lung Reaction Onset: 10:38 AM — approx 23 minutes into infusion at 50 mL/hr \n\nSudden onset chest tightness, facial flushing, urticaria (chest and bilateral upper extremities). Vital signs at reaction: BP\n88/55 mmHg (baseline 122/78), HR 112 bpm, SpO2 94% on room air, Temp 37.8°C. Patient diaphoretic. Severe pruritus\n8/10. Infusion halted immediately.\n\n1. Infusion halted at 10:38 AM — IV line flushed with NS. 2. Epinephrine 0.3 mg IM (right lateral thigh) at 

In [0]:
# PySpark: parse → prep for search

# Read and parse
df = spark.read.format("binaryFile") \
    .load("/Volumes/infusion_center_demo/document_ai/raw_documents/")

parsed_df = df.selectExpr(
    "path",
    "ai_parse_document(content, map('version', '2.0')) AS parsed_content"
)

# Prepare for search
search_ready_df = parsed_df.selectExpr(
    "path",
    "ai_prep_search(parsed_content) AS search_chunks"
)

display(search_ready_df.limit(3))

path search_chunks dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC6_LabResult_CBC_Critical_ANC.pdf {"document":{"contents":[{"chunk_id":"d6952f9e7d844c9f8a3eea6c7b5b1ba6_0","chunk_position":0,"chunk_to_embed":"The following passage represents a chunk of content from a document.\n - 'Content' contains raw document text\n - All other fields describe document context and hierchical information\n - For visual elements like images/charts, a summary is generated as part of 'Content'\n\n Document Title: LABORATORY REPORT — CBC WITH DIFFERENTIAL\n Page Header: \n Page Footer: Ordering provider Dr. Kim notified at 10:22 AM. RECOMMENDATION: Hold ch\n Section Header: \n Caption: \n Footnote: \n Page Number: \n\n Content:\n \nQuest Diagnostics\n\nClinical Laboratory Results\n\nPhone: 1-800-222-0446 | Fax: 1-800-422-1285\n\n1001 Main Street, Teterboro NJ 07608\n\n Patient Name: Alice M. Nguyen Date of Birth: September 5, 1962 MRN: QDX-112233 Ordering Provider: Dr. Rachel Kim, MD — Hematology/Oncology Specimen Collected: January 19, 2025 — 07:30 AM Report Date: January 19, 2025 — 10:15 AM Accession #: 2025-01-19-QD-884421 \n\n TEST RESULT FLAG REFERENCE RANGE UNITS WBC 2.8 LOW ↓ 4.5 – 11.0 K/uL RBC 3.62 LOW ↓ 4.20 – 5.40 M/uL Hemoglobin (HGB) 9.4 LOW ↓ 12.0 – 16.0 g/dL Hematocrit (HCT) 28.1 LOW ↓ 36.0 – 46.0 % Platelets (PLT) 142 150 – 400 K/uL ANC (Neutrophils) 1.1 CRITICAL ↓↓ 1.8 – 7.7 K/uL Lymphocytes 1.3 LOW ↓ 1.0 – 4.8 K/uL Monocytes 0.3 0.2 – 0.9 K/uL ","chunk_to_retrieve":"\nQuest Diagnostics\n\nClinical Laboratory Results\n\nPhone: 1-800-222-0446 | Fax: 1-800-422-1285\n\n1001 Main Street, Teterboro NJ 07608\n\n Patient Name: Alice M. Nguyen Date of Birth: September 5, 1962 MRN: QDX-112233 Ordering Provider: Dr. Rachel Kim, MD — Hematology/Oncology Specimen Collected: January 19, 2025 — 07:30 AM Report Date: January 19, 2025 — 10:15 AM Accession #: 2025-01-19-QD-884421 \n\n TEST RESULT FLAG REFERENCE RANGE UNITS WBC 2.8 LOW ↓ 4.5 – 11.0 K/uL RBC 3.62 LOW ↓ 4.20 – 5.40 M/uL Hemoglobin (HGB) 9.4 LOW ↓ 12.0 – 16.0 g/dL Hematocrit (HCT) 28.1 LOW ↓ 36.0 – 46.0 % Platelets (PLT) 142 150 – 400 K/uL ANC (Neutrophils) 1.1 CRITICAL ↓↓ 1.8 – 7.7 K/uL Lymphocytes 1.3 LOW ↓ 1.0 – 4.8 K/uL Monocytes 0.3 0.2 – 0.9 K/uL \n","pages":[{"image_uri":"","page_id":0}]}],"pages":[{"id":0,"image_uri":null}]},"error_status":null} dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC5_AdverseEvent_Grade3_Pembrolizumab.pdf {"document":{"contents":[{"chunk_id":"b761feebab55475d8d8996c661e14ff3_0","chunk_position":0,"chunk_to_embed":"The following passage represents a chunk of content from a document.\n - 'Content' contains raw document text\n - All other fields describe document context and hierchical information\n - For visual elements like images/charts, a summary is generated as part of 'Content'\n\n Document Title: Infusion Center of North Texas\n Page Header: \n Page Footer: \n Section Header: OUTCOME\n Caption: \n Footnote: \n Page Number: \n\n Content:\n \nAdverse Event / Incident Report\n\n3500 Gaston Avenue, Suite 200, Dallas TX 75246\n\nPhone: 214-820-3300 | Fax: 214-820-3391\n\nharmacist review required\n\nPhysician notified. Mandatory pharmacovigilance reporting initiated.\n\n Report Date/Time: January 18, 2025 — 10:42 AM Patient Name: Elena V. Kowalski Date of Birth: February 14, 1979 MRN: ICNT-334821 Attending Nurse: Jennifer Huang, RN, OCN Drug: Pembrolizumab (Keytruda) 200 mg IV — Cycle 4, Day 1 Diagnosis: C34.12 — Malignant neoplasm of upper lobe, left bronchus/lung Reaction Onset: 10:38 AM — approx 23 minutes into infusion at 50 mL/hr \n\nSudden onset chest tightness, facial flushing, urticaria (chest and bilateral upper extremities). Vital signs at reaction: BP\n88/55 mmHg (baseline 122/78), HR 112 bpm, SpO2 94% on room air, Temp 37.8°C. Patient diaphoretic. Severe pruritus\n8/10. Infusion halted immediately.\n\n1. Infusion halted at 10:38 AM — IV line flushed with NS. 2. Epinephrine 0.3 mg IM (right lateral thigh)

## End-to-End Document Processing Pipeline

**Complete Flow:** PDF Ingestion → Parsing → Classification → Extraction → Search Preparation

This final example shows how all functions work together in a production-ready pipeline.

In [0]:
%sql
-- FULL END-TO-END PIPELINE
-- Parse → Classify → Extract → Prep Search in one query

WITH parsed_docs AS (
  -- Step 1: Parse documents
  SELECT
    path,
    ai_parse_document(content, MAP('version', '2.0')) AS parsed_content
  FROM READ_FILES(
    '/Volumes/infusion_center_demo/document_ai/raw_documents/',
    format => 'binaryFile'
  )
),
classified_docs AS (
  -- Step 2: Classify document types
  SELECT
    path,
    parsed_content,
    ai_classify(
      parsed_content,
      '{
        "patient_record": "Medical records, patient history",
        "prescription": "Medication prescriptions",
        "lab_report": "Lab results and diagnostics",
        "insurance_form": "Insurance and billing",
        "consent_form": "Patient consent forms"
      }',
      MAP('version', '2.0', 'instructions', 'Classify healthcare documents.')
    ) AS document_type
  FROM parsed_docs
),
extracted_docs AS (
  -- Step 3: Extract structured fields
  SELECT
    path,
    parsed_content,
    document_type,
    ai_extract(
      parsed_content,
      '["patient_name", "patient_id", "date_of_service", "diagnosis"]',
      MAP('version', '2.0')
    ) AS extracted_fields
  FROM classified_docs
)
-- Step 4: Prepare for search
SELECT
  path,
  document_type,
  extracted_fields,
  ai_prep_search(parsed_content) AS search_chunks
FROM extracted_docs
LIMIT 5

path document_type extracted_fields search_chunks dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC6_LabResult_CBC_Critical_ANC.pdf {"error_message":null,"response":["lab_report"]} {"error_message":null,"response":{"date_of_service":"January 19, 2025","diagnosis":null,"patient_id":"QDX-112233","patient_name":"Alice M. Nguyen"}} {"document":{"contents":[{"chunk_id":"ae2ba911f5494ecb8fa687e947a95c79_0","chunk_position":0,"chunk_to_embed":"The following passage represents a chunk of content from a document.\n - 'Content' contains raw document text\n - All other fields describe document context and hierchical information\n - For visual elements like images/charts, a summary is generated as part of 'Content'\n\n Document Title: LABORATORY REPORT — CBC WITH DIFFERENTIAL\n Page Header: \n Page Footer: Ordering provider Dr. Kim notified at 10:22 AM. RECOMMENDATION: Hold ch\n Section Header: \n Caption: \n Footnote: \n Page Number: \n\n Content:\n \nQuest Diagnostics\n\nClinical Laboratory Results\n\nPhone: 1-800-222-0446 | Fax: 1-800-422-1285\n\n1001 Main Street, Teterboro NJ 07608\n\n Patient Name: Alice M. Nguyen Date of Birth: September 5, 1962 MRN: QDX-112233 Ordering Provider: Dr. Rachel Kim, MD — Hematology/Oncology Specimen Collected: January 19, 2025 — 07:30 AM Report Date: January 19, 2025 — 10:15 AM Accession #: 2025-01-19-QD-884421 \n\n TEST RESULT FLAG REFERENCE RANGE UNITS WBC 2.8 LOW ↓ 4.5 – 11.0 K/uL RBC 3.62 LOW ↓ 4.20 – 5.40 M/uL Hemoglobin (HGB) 9.4 LOW ↓ 12.0 – 16.0 g/dL Hematocrit (HCT) 28.1 LOW ↓ 36.0 – 46.0 % Platelets (PLT) 142 150 – 400 K/uL ANC (Neutrophils) 1.1 CRITICAL ↓↓ 1.8 – 7.7 K/uL Lymphocytes 1.3 LOW ↓ 1.0 – 4.8 K/uL Monocytes 0.3 0.2 – 0.9 K/uL ","chunk_to_retrieve":"\nQuest Diagnostics\n\nClinical Laboratory Results\n\nPhone: 1-800-222-0446 | Fax: 1-800-422-1285\n\n1001 Main Street, Teterboro NJ 07608\n\n Patient Name: Alice M. Nguyen Date of Birth: September 5, 1962 MRN: QDX-112233 Ordering Provider: Dr. Rachel Kim, MD — Hematology/Oncology Specimen Collected: January 19, 2025 — 07:30 AM Report Date: January 19, 2025 — 10:15 AM Accession #: 2025-01-19-QD-884421 \n\n TEST RESULT FLAG REFERENCE RANGE UNITS WBC 2.8 LOW ↓ 4.5 – 11.0 K/uL RBC 3.62 LOW ↓ 4.20 – 5.40 M/uL Hemoglobin (HGB) 9.4 LOW ↓ 12.0 – 16.0 g/dL Hematocrit (HCT) 28.1 LOW ↓ 36.0 – 46.0 % Platelets (PLT) 142 150 – 400 K/uL ANC (Neutrophils) 1.1 CRITICAL ↓↓ 1.8 – 7.7 K/uL Lymphocytes 1.3 LOW ↓ 1.0 – 4.8 K/uL Monocytes 0.3 0.2 – 0.9 K/uL \n","pages":[{"image_uri":"","page_id":0}]}],"pages":[{"id":0,"image_uri":null}]},"error_status":null} dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC5_AdverseEvent_Grade3_Pembrolizumab.pdf {"error_message":null,"response":["patient_record"]} {"error_message":null,"response":{"date_of_service":"January 18, 2025 — 10:42 AM","diagnosis":"C34.12 — Malignant neoplasm of upper lobe, left bronchus/lung","patient_id":"ICNT-334821","patient_name":"Elena V. Kowalski"}} {"document":{"contents":[{"chunk_id":"90ac15634d274dd5836812c0fcc68a1f_0","chunk_position":0,"chunk_to_embed":"The following passage represents a chunk of content from a document.\n - 'Content' contains raw document text\n - All other fields describe document context and hierchical information\n - For visual elements like images/charts, a summary is generated as part of 'Content'\n\n Document Title: Infusion Center of North Texas\n Page Header: Phone: 214-820-3300 | Fax: 214-820-3391\n Page Footer: \n Section Header: OUTCOME\n Caption: \n Footnote: \n Page Number: \n\n Content:\n \nAdverse Event / Incident Report\n\n3500 Gaston Avenue, Suite 200, Dallas TX 75246\n\nharmacist review required\n\nPhysician notified. Mandatory pharmacovigilance reporting initiated.\n\n Report Date/Time: January 18, 2025 — 10:42 AM Patient Name: Elena V. Kowalski Date of Birth: February 14, 1979 MRN: ICNT-334821 Attending Nurse: Jennifer Huang, RN, OCN Drug: Pembrolizumab (Keytruda) 200 mg IV — Cycle 4, Day 1 Diagnosis: C34.12 — Malignant ne

## Persisting Results

To save your processed documents to a Delta table for querying and analysis:

```sql
CREATE OR REPLACE TABLE infusion_center_demo.document_ai.processed_documents AS
WITH parsed_docs AS (
  SELECT
    path,
    ai_parse_document(content, MAP('version', '2.0')) AS parsed_content
  FROM READ_FILES(
    '/Volumes/infusion_center_demo/document_ai/raw_documents/',
    format => 'binaryFile'
  )
)
SELECT
  path,
  ai_classify(
    parsed_content,
    '{"patient_record": "Medical records", "prescription": "Prescriptions", "lab_report": "Lab reports"}',
    MAP('version', '2.0')
  ) AS document_type,
  ai_extract(
    parsed_content,
    '["patient_name", "patient_id", "date_of_service"]',
    MAP('version', '2.0')
  ) AS extracted_fields,
  ai_prep_search(parsed_content) AS search_chunks
FROM parsed_docs;
```

In [0]:
%sql
-- Create a Delta table with all processed document data
-- This persists: classification, extracted fields, and search chunks

CREATE OR REPLACE TABLE infusion_center_demo.document_ai.processed_documents AS
WITH parsed_docs AS (
  SELECT
    path,
    ai_parse_document(content, MAP('version', '2.0')) AS parsed_content
  FROM READ_FILES(
    '/Volumes/infusion_center_demo/document_ai/raw_documents/',
    format => 'binaryFile'
  )
),
classified_docs AS (
  SELECT
    path,
    parsed_content,
    ai_classify(
      parsed_content,
      '{
        "patient_record": "Medical records, patient history, treatment notes",
        "prescription": "Medication prescriptions and pharmacy orders",
        "lab_report": "Lab results, test reports, diagnostic findings",
        "insurance_form": "Insurance claims, authorization forms, billing",
        "consent_form": "Informed consent, HIPAA forms, patient agreements"
      }',
      MAP('version', '2.0', 'instructions', 'Classify healthcare documents by their primary purpose.')
    ) AS document_type
  FROM parsed_docs
)
SELECT
  path,
  document_type,
  ai_extract(
    parsed_content,
    '{
      "patient_name": {"type": "string", "description": "Full name of the patient"},
      "patient_id": {"type": "string", "description": "Medical record number or patient ID"},
      "date_of_service": {"type": "string", "description": "Date in YYYY-MM-DD format"},
      "diagnosis": {"type": "string", "description": "Primary diagnosis or reason for visit"},
      "medications": {
        "type": "array",
        "description": "List of prescribed medications",
        "items": {
          "type": "object",
          "properties": {
            "name": {"type": "string"},
            "dosage": {"type": "string"},
            "frequency": {"type": "string"}
          }
        }
      }
    }',
    MAP('version', '2.0', 'instructions', 'Extract key patient information from medical documents.')
  ) AS extracted_fields,
  ai_prep_search(parsed_content) AS search_chunks
FROM classified_docs

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Verify the table was created and query the processed documents

SELECT 
  path,
  document_type:response[0] AS doc_category,
  extracted_fields:response:patient_name AS patient_name,
  extracted_fields:response:patient_id AS patient_id,
  extracted_fields:response:date_of_service AS date_of_service,
  extracted_fields:response:diagnosis AS diagnosis,
  size(try_cast(search_chunks:document:contents AS ARRAY<VARIANT>)) AS num_search_chunks
FROM infusion_center_demo.document_ai.processed_documents
ORDER BY path

path,doc_category,patient_name,patient_id,date_of_service,diagnosis,num_search_chunks
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC1_PriorAuth_APPROVAL_Herceptin_BCBS.pdf,"""insurance_form""","""Sarah J. Worthington""","""XYY234567890""",null,"""C50.011 — Malignant neoplasm of central portion, right female breast""",1
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC2_PriorAuth_DENIAL_Rituximab_AETNA.pdf,"""insurance_form""","""James R. Holloway""","""W8834521109""","""2025-01-16""","""M05.79 — Rheumatoid arthritis with rheumatoid factor, multiple sites""",2
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC3_Referral_RCHOP_Oncology_Routine.pdf,"""patient_record""","""Robert A. Martinez""","""TXO-789456""","""2025-01-17""","""C83.39 — Diffuse large B-cell lymphoma, multiple sites""",2
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC4_Referral_Tocilizumab_STAT_Rheumatology.pdf,"""patient_record""","""Marcus T. Williams""","""DRA-556677""","""2025-01-21""","""M06.09 — Rheumatoid arthritis without rheumatoid factor, multiple sites""",1
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC5_AdverseEvent_Grade3_Pembrolizumab.pdf,"""patient_record""","""Elena V. Kowalski""","""ICNT-334821""","""2025-01-18""","""C34.12 — Malignant neoplasm of upper lobe, left bronchus/lung""",1
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC6_LabResult_CBC_Critical_ANC.pdf,"""lab_report""","""Alice M. Nguyen""","""QDX-112233""","""2025-01-19""",null,1
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC7_InsuranceCard_UHC_ChoicePlus.pdf,"""insurance_form""","""Thomas B. Reynolds""","""W1234567890""",null,null,2
dbfs:/Volumes/infusion_center_demo/document_ai/raw_documents/DOC8_AdverseEvent_Grade2_Nausea_Oxaliplatin.pdf,"""patient_record""","""Patricia M. O'Brien""","""ICNT-228844""","""2025-01-22""","""C18.7 — Malignant neoplasm of sigmoid colon""",1


In [0]:
%sql
-- Example analytics query on the processed documents
-- Count documents by type and show average search chunks

SELECT 
  try_cast(document_type:response[0] AS STRING) AS doc_category,
  COUNT(*) AS document_count,
  ROUND(AVG(size(try_cast(search_chunks:document:contents AS ARRAY<VARIANT>))), 1) AS avg_chunks,
  COLLECT_LIST(try_cast(extracted_fields:response:patient_name AS STRING)) AS patient_names
FROM infusion_center_demo.document_ai.processed_documents
GROUP BY try_cast(document_type:response[0] AS STRING)
ORDER BY document_count DESC

doc_category,document_count,avg_chunks,patient_names
patient_record,4,1.3,"List(Patricia M. O'Brien, Elena V. Kowalski, Robert A. Martinez, Marcus T. Williams)"
insurance_form,3,1.7,"List(James R. Holloway, Sarah J. Worthington, Thomas B. Reynolds)"
lab_report,1,1.0,List(Alice M. Nguyen)


## Key Takeaways

### Function Summary

1. **`ai_parse_document()`**
   - Input: BINARY file content
   - Output: VARIANT with structured elements
   - Use: Extract text, tables, figures from PDFs/images

2. **`ai_classify()`**
   - Input: VARIANT or STRING
   - Output: Document category
   - Use: Automatic document routing and organization

3. **`ai_extract()`**
   - Input: VARIANT or STRING
   - Output: Structured fields (typed data)
   - Use: Pull specific information into database schema

4. **`ai_prep_search()`**
   - Input: VARIANT from ai_parse_document
   - Output: Semantic chunks
   - Use: Enable vector search and RAG applications

### Best Practices

* Always use `MAP('version', '2.0')` for ai_parse_document, ai_classify, ai_extract
* Pass VARIANT directly between functions - don't flatten to text
* Use READ_FILES with `format => 'binaryFile'` for documents
* Add descriptions to schemas and labels for better accuracy
* Chain functions in CTEs for readable, maintainable pipelines